In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---


# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 4 &mdash; Fourier Series</h1></center>

Many signals repeat: a heartbeat, the tick of a clock, the square-wave current that a time-domain EM transmitter drives into a loop on
the ground. Fourier's insight was that any such periodic signal can be rebuilt from sines and cosines whose frequencies are whole-number
multiples of one fundamental frequency, so the signal is fully described by a list of complex numbers, one per harmonic. That list is the
first <i>frequency-domain</i> representation in this course, and every later transform grows out of it. In this module we write the complex
and trigonometric Fourier series, use the orthogonality of sines and cosines to compute the coefficients, work examples that end with a
periodic boxcar and the sinc function, watch the Gibbs phenomenon at a jump, and use a Fourier series to solve a driven oscillator from
MATH 225 before returning to the heartbeat.

# Outline
- [4.1 A Periodic Signal: the Heartbeat](#4.1-A-Periodic-Signal:-the-Heartbeat)
- [4.2 Periodicity](#4.2-Periodicity)
- [4.3 Superposition and the Complex Fourier Series](#4.3-Superposition-and-the-Complex-Fourier-Series)
  - [4.3.1 What are the coefficients $C_k$?](#4.3.1-What-are-the-coefficients-$C_k$?)
- [4.4 Magnitude and Phase](#4.4-Magnitude-and-Phase)
- [4.5 The Trigonometric Fourier Series](#4.5-The-Trigonometric-Fourier-Series)
  - [4.5.1 Inversion formula for the trigonometric Fourier series](#4.5.1-Inversion-formula-for-the-trigonometric-Fourier-series)
- [4.6 Orthogonality of Sine and Cosine](#4.6-Orthogonality-of-Sine-and-Cosine)
- [4.7 Worked Examples](#4.7-Worked-Examples)
  - [4.7.1 Example 1: two cosines](#4.7.1-Example-1:-two-cosines)
  - [4.7.2 Example 2: a mixed trigonometric signal](#4.7.2-Example-2:-a-mixed-trigonometric-signal)
  - [4.7.3 Example 3: the periodic boxcar](#4.7.3-Example-3:-the-periodic-boxcar)
  - [4.7.4 The sinc function](#4.7.4-The-sinc-function)
  - [4.7.5 Numerical example and RMS error](#4.7.5-Numerical-example-and-RMS-error)
  - [4.7.6 Thought exercises](#4.7.6-Thought-exercises)
  - [4.7.7 Going to L'Hôpital's rule](#4.7.7-Going-to-L'Hôpital's-rule)
- [4.8 Convergence at Discontinuities: the Gibbs Phenomenon](#4.8-Convergence-at-Discontinuities:-the-Gibbs-Phenomenon)
- [4.9 Fourier Series and ODEs/PDEs](#4.9-Fourier-Series-and-ODEs/PDEs)
- [4.10 The Heartbeat Revisited](#4.10-The-Heartbeat-Revisited)
- [4.11 Parting Thoughts](#4.11-Parting-Thoughts)
- [Microquestions](#Microquestions)

In [ ]:
# Module 4 helpers: audio I/O and a compact audio player
from scipy.io import wavfile
from scipy import signal
from fractions import Fraction
import IPython.display as ipd

def embed_audio(x, rate, max_rate=8000, seconds=None):
    """Embed mono audio, optionally cropped to the first `seconds`, resampled to <= max_rate (keeps the notebook small)."""
    x = np.asarray(x, dtype=float)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if seconds is not None:
        x = x[:int(seconds*rate)]
    if rate > max_rate:
        r = Fraction(max_rate/rate).limit_denominator(1000)
        x = signal.resample_poly(x, r.numerator, r.denominator); rate = rate*r.numerator/r.denominator
    return ipd.Audio(x, rate=rate)

sr, xx = wavfile.read('wav/heartbeat-05.wav')
xx = (xx/np.iinfo(xx.dtype).max).reshape(len(xx), -1)[:, 0]     # float in [-1, 1], mono

<a name="4.1"></a>
## 4.1 A Periodic Signal: the Heartbeat

Let's start out our investigation into the world of Fourier by looking at [Fourier Series](https://en.wikipedia.org/wiki/Fourier_series).  First, to get an idea of the types of periodic signals we can represents with Fourier Series, let's first look at a data example.  The WAV file is an acoustic recording of a heartbeat over a 30s duration.  We are going to both plot the time series and listen to it (which is easy to do as part of the IPython's display function!).

In [ ]:
C = DSP_COLORS
tt = np.arange(len(xx))/sr
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(tt, xx, color=C['time_domain'], lw=0.6)
ax.set_xlim(0, 30); ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude')
plt.tight_layout(); plt.show()

<b>Figure 4.1.</b> A digital recording of a heartbeat, 30 s long. <i>(Data: U. Iowa, <a href="https://theremin.music.uiowa.edu/">theremin.music.uiowa.edu</a>.)</i>

Let's now listen to the first 12 s of the recording. How would you go about characterizing the heart-beat signal?

In [ ]:
embed_audio(xx, rate=sr, seconds=12)     # first 12 s at 8 kHz

The mathematical formulation of **Fourier Series** is used for a variety of purposes.  These can include

* Representing **continuous-time periodic** signals such as the heart beat example from above, the tick-tock of a clock, and geophysical instrument output signals such as EM square waves.


* Expressing **periodic** signals using **harmonically related sinusoids** (i.e., $\sin$ and $\cos$) defined by angular frequencies $\{...,-2\omega_0,-\omega_0,0,\omega_0,2\omega_0,...\}$ where $\omega_0$ is called the **fundamental angular frequency** and only takes on discrete values, i.e.,  $k\omega_0$ where $k$ is an integer and $-\infty<k<\infty$.
    
    
* computing and studying Fourier Series for the purpose of [harmonic analysis](https://en.wikipedia.org/wiki/Harmonic_analysis) such as in vibration analysis or environmental sound monitoring.

The Fourier series represents periodic signals, and it rests on two key concepts: **periodicity** and **superposition**.

<a name="4.2"></a>
## 4.2 Periodicity

A *continuous-time* function $x(t)$ is said to be [periodic](https://en.wikipedia.org/wiki/Periodic_function) if there exists a constant $T>0$ such that:

$$ x(t) = x(t+T), \quad t\in(-\infty,\infty) \tag{1} $$ 

The smallest $T$ for which the relationship in equation (1) holds is called the **fundamental period**.  The **fundamental angular frequency**, $\omega_0$, is related to $T$ by:

$$ \omega_0 = \frac{2\pi}{T} \tag{2}$$

Note that because we have defined $\omega = 2\pi f$, the **fundamental frequency** is given by $f_0=1/T$.  You also have to be careful when people talk about "the frequency".  Normally this means the fundamental frequency (but not always) - so it's good to ask. 

The example below shows that a single $x(t) = \cos 8\pi t$ wave has multiple periodicities; however, the fundamental period is given by $T=0.25$s and thus the fundamental frequency is given by $f_0 = 1/T = 1/(0.25\ \mathrm{s}) = 4$ Hz.

In [ ]:
C = DSP_COLORS
t = np.linspace(-1, 1, 2001)
x = np.cos(8*np.pi*t)
T = 2*np.pi/(8*np.pi)                     # fundamental period = 0.25 s
fig, ax = plt.subplots(figsize=(14, 3.2))
ax.plot(t, x, color=C['time_domain'])
ax.annotate('', xy=(T, 1.18), xytext=(0, 1.18), arrowprops=dict(arrowstyle='<->', color=C['highlight'], lw=1.8))
ax.text(T/2, 1.27, f'one period $T$ = {T:.2f} s', ha='center', va='bottom', color=C['highlight'], fontsize=12)
ax.set_xlim(-1, 1); ax.set_ylim(-1.15, 1.6)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude'); ax.set_title(r'$x(t)=\cos(8\pi t)$')
plt.tight_layout(); plt.show()

<b>Figure 4.2.</b> The signal $x(t)=\cos 8\pi t$ repeats after 0.25 s, 0.5 s, 0.75 s, …; the smallest of these, the fundamental period $T=0.25$ s (orange), gives the fundamental frequency $f_0=1/T=4$ Hz. <i>(Python-generated.)</i>

<a name="4.3"></a>
## 4.3 Superposition and the Complex Fourier Series

From straightforward individual sinusoids we can build up much more complicated periodic waveforms by taking weighted sums over the various sinusoids.  In fact, every continuous-time periodic function can be expanded into a **complex Fourier Series** as:
<div class="alert alert-info">
$$ x(t) = \sum_{k=-\infty}^{\infty} C_k {\rm e}^{i k \omega_0 t}=\sum_{k=-\infty}^{\infty} C_k {\rm e}^{i k 2\pi t/T} \tag{3}$$
</div>

where the **Fourier Series coefficients** $\{C_k\}$ are given by an **inversion formula**:
<div class="alert alert-info">
$$ C_k = \frac{1}{T} \int_{-T/2}^{T/2} x(t) \,{\rm e}^{-i k \omega_0 t} {\rm d}t= \frac{1}{T} \int_{-T/2}^{T/2} x(t) \,{\rm e}^{-i k 2\pi t/T} {\rm d}t, \quad k=...,-2,-1,0,1,2, ...  \tag{4}$$
</div>

<div class="alert alert-info">
<b>Convention:</b> The Fourier series uses $e^{+ik\omega_0 t}$ to build the signal (equation 3) and $\frac{1}{T}\int e^{-ik\omega_0 t}\,dt$ to
analyze it (equation 4), with $\omega_0 = 2\pi/T = 2\pi f_0$: the same sign convention as the Fourier transform pair of Module 3. For a
real signal $x(t)$ the coefficients satisfy $C_{-k}=\overline{C_k}$ (the conjugate of Module 3).
</div>

<div class="alert alert-info">
<b>Synthesis &mdash; complex exponentials as eigenfunctions.</b> Each term $C_k e^{ik\omega_0 t}$ is the rotating phasor of Module 3, turning
$k$ times per period with length $|C_k|$ and starting angle $\angle C_k$. A real signal results because each phasor is paired with its mirror
image $C_{-k}=\overline{C_k}$, which rotates the other way, so the imaginary parts cancel (Module 3: $z+\bar z = 2\Re z$). A Fourier series is
the phasor animation of Module 3 with one phasor per harmonic.
</div>

Note that while ${\rm e}^{i k \omega_0 t}$ may not immediately look like sinusoids, you can apply Euler's formula $\left(\cos\, \theta = \frac{\mathrm{e}^{i\theta}+\mathrm{e}^{-i\theta} }{2} \,{\rm and} \sin\, \theta = \frac{\mathrm{e}^{i\theta}-\mathrm{e}^{-i\theta} }{2i}\right)$ to turn the complex exponential into $\sin$ and $\cos$ terms.  Below is an example of what a Fourier Series would be for a situation where $T=1$s and we have some randomly generated integer coefficients:

$$[...,C_{-1},C_{0},C_1,C_2,C_3,C_4,C_5,...]=[...,0,-5,8,6,4,2,0,...]$$

where all other coefficients are zero.  Figure 4.3 builds this series one term at a time.

In [ ]:
C = DSP_COLORS
t = np.linspace(-2, 2, 2001)
T = 1.0
Ck = [-5, 8, 6, 4, 2]                       # C_0 ... C_4; all other coefficients are zero
terms = [Ck[k]*np.exp(1j*2*np.pi*k*t/T) for k in range(5)]
labels = [r'$-5+8e^{i2\pi t}$', r'$\dots+6e^{i4\pi t}$', r'$\dots+4e^{i6\pi t}$', r'$\dots+2e^{i8\pi t}$']
fig, axs = plt.subplots(2, 2, figsize=(12, 7.5), sharex=True, sharey=True)
for j, ax in enumerate(axs.flat):
    x = sum(terms[:j+2])
    ax.plot(t, x.real, color=C['time_domain'], label='Real')
    ax.plot(t, x.imag, color=C['highlight'], label='Imag')
    ax.set_title(f'({"abcd"[j]}) $C_0$ to $C_{j+1}$: ' + labels[j], fontsize=12)
    ax.set_xlim(-2, 2); ax.set_ylim(-25, 25)
for ax in axs[1]:
    ax.set_xlabel('Time (s)')
for ax in axs[:, 0]:
    ax.set_ylabel('Amplitude')
axs[0, 0].legend(loc='lower left', fontsize=10)
plt.tight_layout(); plt.show()

<b>Figure 4.3.</b> Superposition of (a) 2, (b) 3, (c) 4 and (d) 5 terms of the Fourier series with $T=1$ s and $C_0,\dots,C_4 = -5, 8, 6, 4, 2$: real part (blue) and imaginary part (orange). Adding higher harmonics changes the shape but not the fundamental period $T$. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Note:</b> The harmonics $kf_0$ form the same pattern as the standing-wave modes of a string fixed at both ends in PHGN 200, $f_k = kv/2L$.
A plucked string sounds one pitch $f_0$, and its tone color is set by the amplitudes $|C_k|$ of the higher modes.
</div>

Notice that the shape of the signal repeats with the fundamental periodicity of $T=1$s **regardless of the number of higher-order terms added**! This is a basic illustration of a Fourier series. Of course, we want to be able to represent signals that have a certain shape given by $x(t)$.  In this case we have to use the inversion formula to recover what the $C_k$ values will approximate the desired $x(t)$. 

A natural question might be how many different $\{C_k\}$ coefficients are required to "accurately represent" a given signal?  Generally speaking, this will depend on the signal's structure.  In the most basic example, a single sinusoid will require one coefficient. Similarly, a relatively smooth signal (usually) will require superimposing fewer $\{C_k\}$ terms than a signal with "sharp features" (e.g., a corner discontinuity), which will require a very large (or, in fact, an infinite) number of $\{C_k\}$ terms.  We will look at an example below.

<a name="4.3.1"></a>
### 4.3.1 What are the coefficients $C_k$?

The sequence of coefficients $\{C_k\}$ characterize a **frequency-domain representation** that **uniquely** corresponds to the time-domain signal $x(t)$.  In addition, you should note that time-domain signal in Figure 4.1 is (nearly) both **continuous and periodic**, while the frequency-domain representation is **discrete and aperiodic**!  You will notice that a lot of interesting symmetries occur when looking at **conjugate domains** (i.e., time and frequency).

In [ ]:
C = DSP_COLORS
T = 2.0; w0 = 2*np.pi/T                      # coefficients of Example 2 (Section 4.7.2) on a period T = 2 s
Ck = {-3: (1 - 1j)/(2*np.sqrt(2)), -1: 1 + 0.5j, 0: 1.0, 1: 1 - 0.5j, 3: (1 + 1j)/(2*np.sqrt(2))}
t = np.linspace(-1.5*T, 1.5*T, 3001)
x = sum(c*np.exp(1j*k*w0*t) for k, c in Ck.items()).real
fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw=dict(width_ratios=[1.35, 1]))
axa.plot(t, x, color=C['time_domain'])
for n in (-1, 0, 1):
    axa.axvline(n*T + T/2, color=C['neutral'], ls=':', lw=1)
axa.annotate('', xy=(T/2, -2.35), xytext=(-T/2, -2.35), arrowprops=dict(arrowstyle='<->', color=C['highlight'], lw=1.6))
axa.text(0, -2.2, f'period $T$ = {T:.0f} s', ha='center', va='bottom', color=C['highlight'], fontsize=11)
axa.text(0.03, 0.95, r'$x(t)=\sum_{k=-\infty}^{\infty} C_k\,e^{ik\omega_0 t}$', transform=axa.transAxes, fontsize=13, va='top',
         bbox=dict(fc='white', ec=C['neutral'], boxstyle='round,pad=0.3'))
axa.set_xlim(t[0], t[-1]); axa.set_ylim(-2.6, 6.2)
axa.set_xlabel('Time $t$ (s)'); axa.set_ylabel('$x(t)$'); axa.set_title('(a) Continuous and periodic in time')
ks = np.arange(-4, 5)
mag = np.array([abs(Ck.get(k, 0)) for k in ks])
axb.vlines(ks*w0, 0, mag, color=C['freq_domain'], lw=2)
axb.plot(ks*w0, mag, 'o', color=C['freq_domain'])
axb.axhline(0, color='k', lw=0.8)
axb.annotate('', xy=(w0, 1.3), xytext=(0, 1.3), arrowprops=dict(arrowstyle='<->', color=C['highlight'], lw=1.6))
axb.text(w0/2, 1.36, r'$\omega_0=2\pi/T$', ha='center', va='bottom', color=C['highlight'], fontsize=11)
axb.text(0.03, 0.95, r'$C_k=\frac{1}{T}\int_{-T/2}^{T/2} x(t)\,e^{-ik\omega_0 t}\,dt$', transform=axb.transAxes, fontsize=12,
         va='top', bbox=dict(fc='white', ec=C['neutral'], boxstyle='round,pad=0.3'))
axb.set_xticks(ks*w0); axb.set_xticklabels([f'{k}' if k else '0' for k in ks])
axb.set_xlim(-4.6*w0, 4.6*w0); axb.set_ylim(0, 1.95)
axb.set_xlabel(r'Frequency $\omega = k\omega_0$ (labels: $k$)'); axb.set_ylabel('$|C_k|$')
axb.set_title('(b) Discrete and aperiodic in frequency')
plt.tight_layout(); plt.show()

<b>Figure 4.4.</b> A continuous, periodic signal and its discrete set of Fourier coefficients. (a) The real signal of Example 2 (Section 4.7.2) with period $T=2$ s, built from equation (3). (b) Its coefficient magnitudes $|C_k|$, which exist only at the frequencies $k\omega_0$, spaced $\omega_0 = 2\pi/T$ apart, and are computed from (a) with equation (4). <i>(Python-generated.)</i>

You can also ask yourself: "What happens if I **squeeze** the time series on the left to make $T$ smaller?" Because $\omega_0=2\pi/T$, we see that this effectively **stretches** the frequency content on the right.  That is, it moves the frequency structure higher.  Conversely, if we **stretch** the time series, this will **squeeze** the frequency domain and move the frequencies lower.  

You might note that in the limit that the time series is stretched out to **infinity** (i.e., $T\rightarrow\infty$) this effectively creates a **continuum** in the frequency domain. This limit is the starting point of the **Fourier transform**.

<div class="alert alert-info">
<b>Synthesis &mdash; sampling and periodization.</b> A signal that repeats every $T$ seconds has a spectrum that exists only at the discrete
frequencies $k\omega_0 = 2\pi k/T$. Squeezing the period spreads the lines apart, and stretching it packs them together; as $T\to\infty$ the lines
merge into a continuum. Periodic in one domain, discrete in the other: this pairing is half of a symmetry the course returns to repeatedly.
</div>

<a name="4.4"></a>
## 4.4 Magnitude and Phase

Now that we are back into "complex subject matter", it is helpful to make a connection to the discussion in the previous lectures on complex numbers.  Generally speaking, coefficients $\{C_k\}$ are **complex-valued** quantities, and we often use **magnitude and phase** for its representation.

In [ ]:
C = DSP_COLORS
C1 = 1 - 0.5j                                # C_1 of Example 2
fig, ax = plt.subplots(figsize=(7, 6))
th = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(th), np.sin(th), color=C['unit_circle'], lw=0.8, alpha=0.3)
ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
for c, col, ls, lab in ((C1, C['freq_domain'], '-', r'$C_1=1-\frac{i}{2}$'), (np.conj(C1), C['phase'], '--', r'$C_{-1}=\overline{C_1}=1+\frac{i}{2}$')):
    ax.annotate('', xy=(c.real, c.imag), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color=col, lw=2.2, ls=ls, mutation_scale=16))
    ax.text(c.real + 0.05, c.imag + (0.06 if c.imag > 0 else -0.12), lab, color=col, fontsize=13)
ax.plot([C1.real, C1.real], [0, C1.imag], ':', color=C['neutral']); ax.plot([0, C1.real], [C1.imag, C1.imag], ':', color=C['neutral'])
ax.text(C1.real + 0.03, -0.25, r'$\Re\{C_1\}=1$', fontsize=11, ha='left')
ax.text(-0.05, C1.imag, r'$\Im\{C_1\}=-\frac{1}{2}$', fontsize=11, ha='right', va='center')
ta = np.linspace(0, np.angle(C1), 40)
ax.plot(0.45*np.cos(ta), 0.45*np.sin(ta), color=C['highlight'], lw=2)
ax.text(0.5, -0.13, rf'$\angle C_1$ = {np.degrees(np.angle(C1)):.2f}°', color=C['highlight'], fontsize=11)
ax.text(0.12, -0.43, rf'$|C_1|$ = {abs(C1):.3f}', color=C['freq_domain'], fontsize=11)   # horizontal, below the vector
ax.set_aspect('equal'); ax.set_xlim(-0.75, 1.75); ax.set_ylim(-0.95, 0.95)
ax.set_xlabel('Real part'); ax.set_ylabel('Imaginary part')
plt.tight_layout(); plt.show()

<b>Figure 4.5.</b> Magnitude and phase of a Fourier coefficient. The coefficient $C_1 = 1 - i/2$ of Example 2 (red) has real part 1 and imaginary part $-1/2$, so its magnitude is $|C_1| = 1.118$ and its phase is $\angle C_1 = -26.57^\circ$ (orange). For a real signal $C_{-1} = \overline{C_1}$ (purple, dashed) is its mirror image. Engineering texts often write $j$ for the imaginary unit $i$. <i>(Python-generated.)</i>

Mathematical expressions for these terms are given by the following.  The **magnitude**, always a positive and real quantity, is given by:

$$|C_k| = \sqrt{(\Re{\{C_k}\})^2+(\Im{\{C_k}\})^2} \tag{5}$$

while the **phase** is represented by ($\angle$ denotes angle):

$$\angle (C_k) = \mathrm{atan2}\left(\Im\{C_k\},\, \Re\{C_k\}\right) \tag{6}$$

Thus, the physical meaning of the **Fourier magnitude** and **Fourier phase** is nothing more than the **modulus** and **argument** of a **complex number** that we discussed in the previous chapter. In these expressions $\Re$ and $\Im$ are again the real and imaginary operators:

$$
\Re(z) = \Re(x+iy)=x \quad {\rm and} \quad \Im(z) = \Im(x+iy)=y \tag{7}
$$

As in Module 3, use the four-quadrant $\mathrm{atan2}(y,x)$, not $\tan^{-1}(y/x)$, which cannot tell opposite quadrants apart; in NumPy this is <code>np.angle(C)</code> or <code>np.arctan2(y, x)</code>.

<a name="4.5"></a>
## 4.5 The Trigonometric Fourier Series

There is another Fourier series representation that you will likely encounter involving **trigonometric** $\sin$ and $\cos$ functions:

$$
f(t) = \sum_{k=-\infty}^{\infty} a_k {\rm cos} (k\omega_0 t) + b_k {\rm sin} (k\omega_0 t) \tag{8}
$$

where $a_k$ and $b_k$ are coefficients that are related to the $C_k$ in equations (3) and (4).  Recognizing that the $\cos$ function is even (i.e., ${\rm cos}(-x) = {\rm cos}(x)$) and that the $\sin$ function is odd (i.e., ${\rm sin} (-x) = - {\rm sin} (x)$), one may rewrite equation (8) as: 

<div class="alert alert-info">
$$
f(t) = \frac{A_0}{2} + \sum_{k=1}^{\infty} A_k {\rm cos} (k\omega_0 t) + B_k {\rm sin} (k\omega_0 t) \tag{9}
$$
</div>
where, in general, $A_k \neq a_k$ and $B_k \neq b_k$.  

**Extension Question:** Show that equations (8) and (9) are equivalent by establishing the relationship between the two sets of coefficients.

<a name="4.5.1"></a>
### 4.5.1 Inversion formula for the trigonometric Fourier series

**Fourier series**  consisting of only $\cos$ and $\sin$ terms will occur for even and odd waveforms, respectively.  Waveforms that are neither even nor odd generally will have both $\cos$ and $\sin$ terms.  The **inversion formula** for the Fourier series coefficients are given by

<div class="alert alert-info">
$$
\begin{eqnarray}
A_0 &=& \frac{2}{T}\int_{-T/2}^{T/2} f(t) \,{\rm d}t  \tag{10}\\
A_k &=& \frac{2}{T}\int_{-T/2}^{T/2} f(t) \,{\rm cos}(k\omega_0 t) \,{\rm d}t \tag{11} \\
B_k &=& \frac{2}{T}\int_{-T/2}^{T/2} f(t) \,{\rm sin}(k\omega_0 t) \, {\rm d}t \tag{12}
\end{eqnarray}
$$
</div>

One can establish the equivalence of the trigonometric and complex exponential Fourier Series by applying **Euler's formula** (Module 3).

<a name="4.6"></a>
## 4.6 Orthogonality of Sine and Cosine

An important question that comes to mind when looking at equations (10)–(12) is: "what happens if $f(t)$ is a superposition of several sinusoids?".  To answer this question directly we can turn to the  [orthogonality](https://en.wikipedia.org/wiki/Orthogonal_functions) of $\cos$ and $\sin$ integrals.

**Q:** Prove the orthogonality relationships of $\cos$ and $\sin$ functions over $[-T/2,T/2]$ where $T$ is the fundamental period.

**A:** For any $n\neq0$ and with $\omega_0=\frac{2\pi}{T}$ we have

$$ \int_{-T/2}^{T/2} \mathrm{cos}\,(n\omega_0 t) \,\mathrm{d}t = \left[ \frac{\mathrm{sin}\,(n\omega_0 t)}{n\omega_0}  \right]_{-T/2}^{T/2}=0 \tag{13}$$

$$ \int_{-T/2}^{T/2} \mathrm{sin}\,(n\omega_0 t) \,\mathrm{d}t = \left[- \frac{\mathrm{cos}\,(n\omega_0 t)}{n\omega_0}  \right]_{-T/2}^{T/2}=0 \tag{14}$$

$$ \int_{-T/2}^{T/2} \mathrm{cos}^2(n\omega_0 t) \,\mathrm{d}t = 2\int_{0}^{T/2} \frac{1+\mathrm{cos}(2n\omega_0 t)}{2} \,\mathrm{d}t = \left[ t+\frac{\mathrm{sin}\,(2 n\omega_0 t)}{2 n\omega_0}  \right]_{0}^{T/2}= \frac{T}{2} \tag{15}$$

$$ \int_{-T/2}^{T/2} \mathrm{sin}^2(n\omega_0 t) \,\mathrm{d}t = 2\int_{0}^{T/2} \frac{1-\mathrm{cos}(2n\omega_0 t)}{2} \,\mathrm{d}t = \left[ t-\frac{\mathrm{sin}\,(2 n\omega_0 t)}{2 n\omega_0}  \right]_{0}^{T/2}= \frac{T}{2} \tag{16}$$

Equations (20)–(22) below use the following three identities:

$$\mathrm{cos}(\alpha)\,\mathrm{cos}(\beta) = \frac{1}{2}\left[\mathrm{cos}(\alpha+\beta)+\mathrm{cos}(\alpha-\beta)\right] \tag{17} $$

$$\mathrm{sin}(\alpha)\,\mathrm{sin}(\beta)  = \frac{1}{2}\left[\mathrm{cos}(\alpha-\beta)-\mathrm{cos}(\alpha+\beta)\right] \tag{18} $$

$$\mathrm{sin}(\alpha)\,\mathrm{cos}(\beta)  = \frac{1}{2}\left[\mathrm{sin}(\alpha+\beta)+\mathrm{sin}(\alpha-\beta)\right] \tag{19}$$

For $n\neq m$ (and $n,m\neq0$) we have

$$ \int_{-T/2}^{T/2} \mathrm{cos}(m\omega_0 t) \,\mathrm{cos}(n\omega_0 t) \,\mathrm{d}t = \frac{1}{2}\int_{-T/2}^{T/2} \left[\mathrm{cos}((m+n)\omega_0 t)+\mathrm{cos}((m-n)\omega_0 t)\right] \,\mathrm{d}t = 0 \tag{20}$$

$$ \int_{-T/2}^{T/2} \mathrm{sin}(m\omega_0 t) \,\mathrm{sin}(n\omega_0 t) \,\mathrm{d}t = \frac{1}{2}\int_{-T/2}^{T/2} \left[\mathrm{cos}((m-n)\omega_0 t)-\mathrm{cos}((m+n)\omega_0 t)\right] \,\mathrm{d}t = 0 \tag{21}$$

$$ \int_{-T/2}^{T/2} \mathrm{cos}(m\omega_0 t) \,\mathrm{sin}(n\omega_0 t) \,\mathrm{d}t = \frac{1}{2}\int_{-T/2}^{T/2} \left[\mathrm{sin}((m+n)\omega_0 t)+\mathrm{sin}((m-n)\omega_0 t)\right] \,\mathrm{d}t = 0 \tag{22}$$

Now let's create a numerical test to help confirm/validate our analytic approach! To do this, let's use [numerical quadrature](https://en.wikipedia.org/wiki/Numerical_integration) functions of the Scipy package.

In [ ]:
# Numerical check of the orthogonality integrals (eqs. 20-22 above) with scipy's adaptive quadrature
from scipy.integrate import quad
m, n = 3, 4
ans, err = quad(lambda x: np.cos(m*x)*np.cos(n*x), -np.pi, np.pi); print('The cos-cos integral result is:', ans)
ans, err = quad(lambda x: np.sin(m*x)*np.sin(n*x), -np.pi, np.pi); print('The sin-sin integral result is: %f' % ans)
ans, err = quad(lambda x: np.cos(m*x)*np.sin(n*x), -np.pi, np.pi); print('The cos-sin integral result is: %e' % ans)

Let's also investigate this graphically in Figure 4.6.

In [ ]:
C = DSP_COLORS
m, n = 3, 4                                  # integers: rerun with other values
t = np.linspace(-np.pi, np.pi, 4001)
pairs = [(np.sin(m*t), np.sin(n*t), rf'$\sin {m}t$', rf'$\sin {n}t$', '(a) sin–sin'),
         (np.cos(m*t), np.cos(n*t), rf'$\cos {m}t$', rf'$\cos {n}t$', '(b) cos–cos'),
         (np.cos(m*t), np.sin(n*t), rf'$\cos {m}t$', rf'$\sin {n}t$', '(c) cos–sin')]
fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for ax, (f1, f2, l1, l2, ttl) in zip(axs, pairs):
    ax.fill_between(t, f1*f2, color=C['impulse_resp'], alpha=0.3, lw=0)
    ax.plot(t, f1, color=C['time_domain'], lw=1.6, label=l1)
    ax.plot(t, f2, color=C['highlight'], lw=1.6, label=l2)
    ax.plot(t, f1*f2, color='k', lw=2, label='product')
    ax.set_title(f'{ttl}: integral of the product over one period = {np.trapezoid(f1*f2, t):.1e}', fontsize=12)
    ax.set_ylim(-1.3, 1.3); ax.set_ylabel('Amplitude'); ax.legend(loc='upper right', fontsize=10, ncol=3)
axs[-1].set_xlabel('Time (s)'); axs[-1].set_xlim(-np.pi, np.pi)
plt.tight_layout(); plt.show()

<b>Figure 4.6.</b> Orthogonality of sines and cosines over one period, for $m=3$ and $n=4$: (a) $\sin 3t$ and $\sin 4t$; (b) $\cos 3t$ and $\cos 4t$; (c) $\cos 3t$ and $\sin 4t$. The two functions are blue and orange, their product black, and the shaded area under the product is the integral, which is zero in every case (panel titles). It is instructive to rerun the code with other integers $m$ and $n$. <i>(Python-generated.)</i>

<a name="4.7"></a>
## 4.7 Worked Examples

<a name="4.7.1"></a>
### 4.7.1 Example 1: two cosines

**Q:** Find the **complex exponential Fourier series coefficients** for $x(t)={\rm cos}(10\,\pi\,t)+{\rm cos}(20\,\pi\,t)$:

In [ ]:
C = DSP_COLORS
t = np.linspace(-1, 1, 2001)
x = np.cos(10*np.pi*t) + np.cos(20*np.pi*t)
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, x, color=C['time_domain'])
ax.set_xlim(-1, 1); ax.set_ylim(-1.4, 2.2)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude'); ax.set_title(r'$x(t)=\cos(10\pi t)+\cos(20\pi t)$')
plt.tight_layout(); plt.show()

<b>Figure 4.7.</b> The superposition $x(t)=\cos 10\pi t+\cos 20\pi t$. <i>(Python-generated.)</i>

**A:** It is clear that the fundamental angular frequency of $x(t)$ is $\omega_0=10\pi$ (or $f_0=5$ Hz).  The fundamental period is thus equal to $T=2\pi/\omega_0=1/5$s, which is validated:

$$
\begin{eqnarray}
x\left(t+\frac{n}{5}\right) & = & {\rm cos} \left(10\pi\left(t+\frac{n}{5}\right)\right)+{\rm cos} \left(20\pi\left(t+\frac{n}{5}\right)\right), n=\pm 1,2,3,...\\
\, &=& {\rm cos} \left(10\pi t + 2\pi n \right) + {\rm cos} \left(20\pi t+4\pi n\right), n=\pm 1,2,3,... \tag{23}\\
\, &=& {\rm cos} \left(10\pi t\right) + {\rm cos} \left( 20\pi t\right) = x(t)
\end{eqnarray}
$$

and therefore satisfies the periodicity requirement.  **NOTE: This is a good test to remember because it is a necessary test to demonstrate that $T$ is the fundamental period.**  Now with the use of Euler formulae, we can express $x(t)$ as:

$$
\begin{eqnarray}
x(t) & = & {\rm cos}(10\pi t)+{\rm cos}(20\pi t) = {\rm cos}(\omega_0 t)+{\rm cos}(2\omega_0 t) \\
\,   & = & \frac{{\rm e}^{i\omega_0 t} + {\rm e}^{-i \omega_0 t}}{2} +\frac{{\rm e}^{2 i\omega_0 t} + {\rm e}^{-2i\omega_0 t}}{2} \\
\, & = & \frac{1}{2}{\rm e}^{-2i\omega_0 t}+\frac{1}{2}{\rm e}^{-i\omega_0 t}+\frac{1}{2}{\rm e}^{i\omega_0 t}+\frac{1}{2}{\rm e}^{2i\omega_0 t} = \sum_{k=-\infty}^{\infty} C_ke^{ik\omega_0t}
\end{eqnarray}
\tag{24}
$$

By inspection and use of equation (3) (i.e., the last equality in equation 24), we have $C_{-2}=C_{-1}=C_1=C_2=1/2$.  All other $C_k$ are equal to zero.

In [ ]:
C = DSP_COLORS
f0 = 5.0                                     # fundamental frequency (Hz)
k = np.arange(-10, 11)
Ck = np.where(np.isin(np.abs(k), [1, 2]), 0.5, 0.0)
fig, ax = plt.subplots(figsize=(14, 3))
ax.vlines(k*f0, 0, Ck, color=C['freq_domain'], lw=2)
ax.plot(k*f0, Ck, 'o', color=C['freq_domain'])
ax.axhline(0, color='k', lw=0.8)
ax.set_xlim(-52, 52); ax.set_ylim(-0.05, 0.62)
ax.set_xlabel(r'Frequency $k f_0$ (Hz)'); ax.set_ylabel(r'$C_k$')
plt.tight_layout(); plt.show()

<b>Figure 4.8.</b> The Fourier coefficients of the signal in Figure 4.7: $C_{\pm1}=C_{\pm2}=1/2$ at $\pm5$ Hz and $\pm10$ Hz, and zero elsewhere. <i>(Python-generated.)</i>

<a name="4.7.2"></a>
### 4.7.2 Example 2: a mixed trigonometric signal

**Q:** Find the **complex exponential Fourier Series** coefficients for 

$$x(t) = 1+{\rm sin} (\omega_0 t)+2{\rm cos} (\omega_0 t)+{\rm cos} (3\omega_0 t+\pi/4)$$

In [ ]:
C = DSP_COLORS
w0 = np.pi                                   # fundamental angular frequency (rad/s): period 2 s
xf = lambda t: 1 + np.sin(w0*t) + 2*np.cos(w0*t) + np.cos(3*w0*t + np.pi/4)
t = np.linspace(-2.5*np.pi, 2.5*np.pi, 4001)
ts = np.arange(-2.5*np.pi, 2.5*np.pi, 0.1)
fig, ax = plt.subplots(figsize=(14, 3.2))
ax.plot(t, xf(t), color=C['continuous'], label='continuous')
ax.plot(ts, xf(ts), 'x', color=C['sampled'], ms=5, label='samples every 0.1 s')
ax.set_xlim(t[0], t[-1]); ax.set_ylim(-2.6, 4.9)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude'); ax.legend(loc='upper right', fontsize=10, ncol=2)
plt.tight_layout(); plt.show()

<b>Figure 4.9.</b> The signal $x(t)=1+\sin\omega_0 t+2\cos\omega_0 t+\cos(3\omega_0 t+\pi/4)$ with $\omega_0=\pi$ rad/s, in continuous form (gray) and sampled every 0.1 s (blue crosses). <i>(Python-generated.)</i>

**A:** With use of Euler formulae, we can expand out each term in the function $x(t)$ to write:

$$
\begin{eqnarray}
x(t) &=& 1+\sin(\omega_0 t)+2 \cos(\omega_0 t)+\cos(3\omega_0 t+\pi/4) \tag{25}
\\
&=& 1 + \frac{e^{i\omega_0t}-e^{-i\omega_0t}}{2i}+e^{i\omega_0t}+e^{-i\omega_0t}+
\frac{e^{i(3\omega_0t+\pi/4)}+e^{-i(3\omega_0t+\pi/4)}}{2} \tag{26}
\\
&=& 1+\left(1+\frac{1}{2i}\right){\rm e}^{i\omega_0 t}+
\left(1-\frac{1}{2i}\right){\rm e}^{-i\omega_0 t}+
\frac{1}{2} {\rm e}^{i\pi/4}{\rm e}^{3i\omega_0 t}+
\frac{1}{2} {\rm e}^{-i\pi/4}{\rm e}^{-3i\omega_0 t} \tag{27}
\\
\, & = &
\frac{1}{2\sqrt{2}}\left(1-i\right){\rm e}^{-3 i \omega_0 t}+
\left(1+\frac{i}{2}\right){\rm e}^{-i\omega_0 t}+
1 +
\left(1-\frac{i}{2}\right){\rm e}^{i\omega_0 t}+
\frac{1}{2\sqrt{2}}\left(1+i\right) {\rm e}^{3 i \omega_0 t} \tag{28}
\end{eqnarray}
$$
Thus, using equation (3) we have
$$
C_k =
\left\{
\begin{array}{ll}
\frac{1}{2\sqrt{2}}\left(1-i\right), & k=-3\\
1+\frac{i}{2}, & k=-1 \\
1,  & k=0\\
1-\frac{i}{2}, & k=1 \\
\frac{1}{2\sqrt{2}}\left(1+i\right), & k=3\\
0 & {\rm otherwise}
\end{array}
\right.
\tag{29}
$$

**Q:** What do you notice about the coefficients?

**A:** Symmetry (but why?)! Check that $\overline{C_k} = C_{-k}$.  To plot $\{C_k\}$ we need to compute $|C_k|$ and $\angle(C_k)$ for all $k$.

In [ ]:
C = DSP_COLORS
k = np.arange(-5, 6)
Ck = np.zeros(k.size, dtype=complex)
Ck[k == -3] = np.sqrt(2)/4*(1 - 1j); Ck[k == -1] = 1 + 0.5j; Ck[k == 0] = 1
Ck[k == 1] = 1 - 0.5j;              Ck[k == 3] = np.sqrt(2)/4*(1 + 1j)
mag, ang = np.abs(Ck), np.angle(Ck)
fig, (axa, axb) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axa.vlines(k, 0, mag, color=C['freq_domain'], lw=2); axa.plot(k, mag, 'o', color=C['freq_domain'])
axa.axhline(0, color='k', lw=0.8); axa.set_ylim(-0.05, 1.3); axa.set_ylabel(r'$|C_k|$'); axa.set_title('(a) Magnitude')
axb.vlines(k, 0, ang, color=C['phase'], lw=2); axb.plot(k, ang, 'o', color=C['phase'])
axb.axhline(0, color='k', lw=0.8); axb.set_ylim(-1.05, 1.05); axb.set_ylabel(r'$\angle C_k$ (rad)'); axb.set_title('(b) Phase')
axb.set_xlabel('$k$'); axb.set_xlim(-5.5, 5.5); axb.set_xticks(k)
plt.tight_layout(); plt.show()

<b>Figure 4.10.</b> The (a) magnitude and (b) phase (in radians) of the Fourier coefficients of the signal in Figure 4.9, from equation (29). Note the symmetry: $|C_{-k}|=|C_k|$ and $\angle C_{-k} = -\angle C_k$. <i>(Python-generated.)</i>

<a name="4.7.3"></a>
### 4.7.3 Example 3: the periodic boxcar

**Q:** Find the **complex exponential Fourier series** coefficients for $x(t)$, which is a periodic continuous-time signal of fundamental period $T$ and is a pulse with a width of $T_0$ in each period. Over the specific period from $-T/2$ to $T/2$, $x(t)$ is:

$$
x(t) = \Pi_{T_0/2}(t) =
\left\{
\begin{array}{ll}
1, & -T_0/2 < t < T_0/2\\
0, & {\rm otherwise}
\end{array}
\right.
\tag{30}
$$

with $T>T_0$.  Note that this function is commonly denoted $\Pi_a$, where $a$ is the <i>half-width</i> of the "boxcar" (the convention used throughout this course), so a pulse of width $T_0$ is $\Pi_{T_0/2}$.  This function is sometimes called the **top hat** or **rect** function.

In [ ]:
C = DSP_COLORS
T, T0 = 2.0, 1.0                             # period and pulse width (half-width T0/2)
t = np.linspace(-3, 3, 6001)
x = (np.abs((t + T/2) % T - T/2) < T0/2).astype(float)
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, x, color=C['time_domain'])
ax.annotate('', xy=(T0/2, 1.12), xytext=(-T0/2, 1.12), arrowprops=dict(arrowstyle='<->', color=C['highlight'], lw=1.6))
ax.text(0, 1.17, f'$T_0$ = {T0:.0f} s', ha='center', va='bottom', color=C['highlight'], fontsize=11)
ax.annotate('', xy=(T, -0.15), xytext=(0, -0.15), arrowprops=dict(arrowstyle='<->', color=C['neutral'], lw=1.6))
ax.text(T/2, -0.22, f'$T$ = {T:.0f} s', ha='center', va='top', color=C['neutral'], fontsize=11)
ax.set_xlim(-3, 3); ax.set_ylim(-0.5, 1.45)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude')
plt.tight_layout(); plt.show()

<b>Figure 4.11.</b> Three periods of the periodic boxcar with period $T=2$ s and pulse width $T_0=1$ s. <i>(Python-generated.)</i>

**A:** Using the **inversion formula** for the complex Fourier series (i.e., equation 4) we obtain:

$$
C_k = \frac{1}{T} \int_{-T/2}^{T/2} x(t) {\rm e}^{-ik\omega_0 t} {\rm d}t = \frac{1}{T}\int_{-T_0/2}^{T_0/2} {\rm e}^{-ik\omega_0 t} {\rm d}t \tag{31}
$$

For $k=0$: We can immediately set the complex exponential term in equation (31) equal to one, which leaves the following integral:

$$
C_0 = \frac{1}{T}\int_{-T_0/2}^{T_0/2} {\rm d}t = \frac{t}{T} \left|_{-T_0/2}^{T_0/2}\right. = \frac{T_0/2-(-T_0/2)}{T} = \frac{T_0}{T} \tag{32}
$$

For $k\neq0$: We need to evaluate the full integral.

$$
C_k = \frac{1}{T}\int_{-T_0/2}^{T_0/2} {\rm e}^{-ik\omega_0 t} = -{\frac{1}{ik\omega_0 T}{\rm e}^{-ik\omega_0t}}\left|_{-T_0/2}^{T_0/2}\right. =  
\frac{{\rm e}^{ik\omega_0 T_0/2}-{\rm e}^{-ik\omega_0 T_0/2}}{ik\omega_0 T}  = 
\frac{2{\rm sin}(k\omega_0 T_0/2)}{k\omega_0 T}  \tag{33}
$$

where we have used the Euler expression for $\sin$ in the last step. Let's now rewrite $\omega_0$ in terms of $2\pi/T$

$$
C_k = 
\frac{2{\rm sin}(k\omega_0 T_0/2)}{k\omega_0 T}  = 
\frac{2T{\rm sin}(\pi k T_0/T)}{2\pi k T}  = 
\frac{{\rm sin} (\pi k T_0/T)}{\pi k} = \frac{T_0}{T}\,\mathrm{sinc}\!\left(\frac{kT_0}{T}\right) \tag{34}
$$

Now that we have the coefficients for the complex Fourier series, we can use equation (3) to generate the final expression:

$$
f(t) = \frac{T_0}{T} + \sum_{k=-\infty \,(k\neq0)}^{\infty} \frac{{\rm sin} (\pi k T_0/T)}{k\pi} {\rm e}^{i2 k \pi t/T} \tag{35}
$$

You may also have noticed that we can break up the full sum in equation (35) by using two different sums, one positive and one negative, which allows us to do the following mathematical manipulation:  

$$
\begin{eqnarray}
f(t) &=& 
\frac{T_0}{T} + \sum_{k=-\infty\,(k\neq0)}^{\infty} \frac{{\rm sin} (\pi k T_0/T)}{k\pi} {\rm e}^{i2 k \pi t/T} \tag{36} \\
&=& 
\frac{T_0}{T} + \sum_{k=-\infty}^{-1} \frac{{\rm sin} (\pi k T_0/T)}{k\pi} {\rm e}^{i2 k \pi t/T}
+
\sum_{k=1}^{\infty} \frac{{\rm sin} (\pi k T_0/T)}{k\pi} {\rm e}^{i2 k \pi t/T} \tag{37}\\
&=&
\frac{T_0}{T} + \sum_{k=1}^{\infty} \frac{{\rm sin} (-\pi k T_0/T)}{-k\pi} {\rm e}^{-i2 k \pi t/T}
+
\sum_{k=1}^{\infty} \frac{{\rm sin} (\pi k T_0/T)}{k\pi} {\rm e}^{i2 k \pi t/T} \tag{38}\\
&=&
\frac{T_0}{T} + \sum_{k=1}^{\infty} \frac{{\rm sin} (\pi k T_0/T)}{k\pi} {\rm e}^{-i2 k \pi t/T}
+
\sum_{k=1}^{\infty} \frac{{\rm sin} (\pi k T_0/T)}{k\pi} {\rm e}^{i2 k \pi t/T} \tag{39}\\
&=&
\frac{T_0}{T} + \sum_{k=1}^{\infty} \frac{{\rm sin} (\pi k T_0/T)}{k\pi}  \left[ {\rm e}^{i2 k \pi t/T} +{\rm e}^{-i2 k \pi t/T}  \right]\tag{40} \\
&=&\frac{T_0}{T} + 2 \sum_{k=1}^{\infty} \frac{{\rm sin} (\pi k T_0/T)}{k\pi}  \cos (2\pi k t/T)
\tag{41}
\end{eqnarray}
$$

Interestingly, this complex Fourier series also can be written purely with $\cos$ terms, which makes sense because $f(t)$, $\cos$, and $\frac{\sin(\pi k T_0/T)}{k\pi}$ are all even functions!  Thus, one could have easily used a trigonometric Fourier Series and immediate set all of the $\sin$ terms to zero.

<a name="4.7.4"></a>
### 4.7.4 The sinc function

As a check, set $T_0 = T$ in equation (35) (or 41): the pulse fills the whole period, so $x(t)$ is the constant 1, and indeed

$$
f(t) = 1 + \sum_{k=-\infty\,(k\neq0)}^{\infty} \frac{{\rm sin}\, k\pi}{k\pi} {\rm e}^{i2 k \pi t/T} = 1 \tag{42}
$$
because $\sin k\pi = 0$ for every integer $k \neq 0$.

You will encounter the following expression quite a few times in this course (and elsewhere):

$$ \mathrm{sinc}(x) \equiv \frac{\sin \pi x}{\pi x}, \qquad \mathrm{sinc}(0)=1 \tag{43} $$

This is the [normalized sinc function](https://en.wikipedia.org/wiki/Sinc_function), used throughout this course; it is zero at every nonzero integer $x$ and is very important in low-pass filtering and interpolation. With it, the boxcar coefficients of equation (34) are simply $C_k = (T_0/T)\,\mathrm{sinc}(kT_0/T)$.

<div class="alert alert-warning">
<b>Common misconception:</b> "sinc$(x)$ means $\sin x / x$." Some texts use this <i>unnormalized</i> form, whose zeros
fall at multiples of $\pi$ rather than at the integers. NumPy's <code>np.sinc(x)</code> is the normalized $\sin(\pi x)/(\pi x)$ of equation (43),
so <code>np.sinc(x/np.pi)</code> gives $\sin x/x$. Always check which definition a formula assumes.
</div>

<a name="4.7.5"></a>
### 4.7.5 Numerical example and RMS error

Figure 4.12 plots this example for $T=2.0$ s and $T_0=1.0$ s (i.e., from -0.5s to 0.5s).

In [ ]:
C = DSP_COLORS
dt = 0.01
t = np.arange(-1.0, 1.01, dt)
nc = 1111                                    # number of coefficients: try other values (QUESTIONS in Section 4.8)
T, T0 = 2.0, 1.0
# the boxcar being approximated, with the half-values at the two jumps
x = np.zeros(t.size)
x[51:150] = 1.
x[50] = 0.5
x[150] = 0.5
# eq. (36f): y = T0/T + 2 sum_{k>=1} sin(pi k T0/T)/(pi k) cos(2 pi k t/T)
k = np.arange(1, nc)
Ck = np.sin(np.pi*k*T0/T)/(np.pi*k)
y = T0/T + 2*np.sum(Ck[:, None]*np.cos(2*np.pi*k[:, None]*t/T), axis=0)
err = np.abs(y - x)
fig, (axa, axb) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axa.plot(t, x, color=C['continuous'], lw=3, label='boxcar $x(t)$')
axa.plot(t, y, color=C['time_domain'], lw=1.5, label=f'Fourier series, {nc - 1} terms per side')
axa.set_ylim(-0.2, 1.3); axa.set_ylabel('Amplitude'); axa.legend(loc='upper right', fontsize=10)
axa.set_title(f'(a) $T$ = {T:.0f} s and $T_0$ = {T0:.0f} s')
axb.plot(t, err, color=C['freq_domain'])
axb.set_ylabel(r'$\epsilon = |y - x|$'); axb.set_xlabel('Time (s)'); axb.set_title('(b) Approximation error')
axb.set_xlim(-1, 1)
plt.tight_layout(); plt.show()

<b>Figure 4.12.</b> (a) The boxcar of Figure 4.11 (gray), with the value 1/2 at the two jumps, and its Fourier-series approximation (blue) with 1110 terms on each side of $k=0$. (b) The approximation error $|y-x|$, largest next to the jumps. The normalized RMS error (below) is 0.20%; rerun with the half-values removed (<code>x[50:150] = 1.</code>) and it rises to 7.07%. The accuracy also changes with the number of coefficients <code>nc</code>. <i>(Python-generated.)</i>

For the second plot we have defined the **residual error** (as a function of time) by computing $\epsilon=|y-x|$, where $x(t)$ is the original signal and $y(t)$ is our Fourier Series approximation of it.  Importantly, this error measure allows us to visually inspect how well we are doing.  

Note that we could also define a **normalized root-mean-square (RMS) error** that is a unitless ratio of the **RMS error** to the **RMS signal**:

$$ \epsilon_{RMS} = \sqrt{ \frac{\sum_{i=0}^{N-1}(y_i-x_i)^2}{\sum_{i=0}^{N-1} x_i^2}} \tag{44}$$

Let's compute that for the given number of coefficients:

In [ ]:
error = 100.*np.sqrt(np.sum((y - x)**2)/np.sum(x**2))
print(f'The Fourier series approximation percentage error using {nc} coefficients is: {error:.3f} %')

<a name="4.7.6"></a>
### 4.7.6 Thought exercises

Which type of Fourier Series would you use (i.e., complex or trigonometric) for the following periodic functions and why? Are there any integrals that you can just state are equal to zero and why? Are there any non-zero integrals that you can solve by inspection and why?

1. Saw tooth 

$$s(t) = -\frac{t}{T}, \quad  -\frac{T}{2} < t < \frac{T}{2}$$

2. Absolute-value saw-tooth

$$s(t) = -\frac{|t|}{T}, \quad -\frac{T}{2} < t < \frac{T}{2}$$

3. Exponential Decay ($\alpha$ is a positive real value)

$$s(t) = \left\{  
\begin{array}{cc}
0, \quad & -T/2 < t \le 0 \\
e^{-\alpha t}, \quad & 0 < t \le T/2 \\
\end{array}
\right. $$

4. Absolute value exponential decay ($\alpha$ is a positive real value)

$$s(t) = e^{-\alpha|t|}, \quad  -\frac{T}{2} < t < \frac{T}{2}$$

5. Shifting series

$$s(t) = \left\{  
\begin{array}{cc}
-1/2, \quad & -T/2 < t \le -T/4 \\
1/2, \quad & -T/4 < t \le T/4 \\
-1/2, \quad & T/4 < t \le T/2 \\
\end{array}
\right. $$

<a name="4.7.7"></a>
### 4.7.7 Going to L'Hôpital's rule

Note that in equations (31)–(33) we explicitly separated the $k=0$ case from the $k\neq0$ case to make it easier to evaluate this coefficient. However, some of you may be concerned about what happens if we were to evaluate equation (34) with a $k=0$ in the denominator.  

It turns out that we can still derive the $k=0$ term from the $k\neq0$ expression using [L'H$\hat{{\rm o}}$pital's rule](https://en.wikipedia.org/wiki/L%27H%C3%B4pital%27s_rule):

$$\lim _{x\to c}{\frac {f(x)}{g(x)}}=\lim _{x\to c}{\frac {f'(x)}{g'(x)}} \tag{45}$$

Applying this rule to the expression in equation (34) gives:

$$
\lim_{k \rightarrow 0} \frac{{\rm sin} (\pi k T_0/T)}{k\pi} 
= \lim_{k\rightarrow0} 
\frac{ 
\frac{{\rm d}}{{\rm d}k} {\rm sin} (\pi k T_0/T)
}
{\frac{{\rm d}}{{\rm d}k} k\pi}
=
\lim_{k \rightarrow 0} \frac{\pi T_0/T {\rm cos}(\pi k T_0/T)}{\pi}
=
\frac{T_0}{T} \tag{46}
$$

which is identical to equation (32) above.

<a name="4.8"></a>
## 4.8 Convergence at Discontinuities: the Gibbs Phenomenon

Let's ask ourselves a few questions about why we are getting such significant error at the location of the discontinuities.  

**QUESTIONS:**

1. Repeat the above numerical test using a different number of terms in the series (e.g., nc=51, nc=101, nc=201, nc=501 and nc=1001). 

2. Comment on the differences and how this changes the approximation error.  

3. What happens if you define your box car as follows?
   * x[50:150]=1. 
   * x[50]=0.5 
   * x[150]=0.5 
   
4. How many terms are required to have an approximation error $|\epsilon|<1$%?

The main reason you are seeing the approximation error not go to zero is that a Fourier series converges to the function $\overline{f}$, which is equal to the original function at **points of continuity** or to the average of the two limits at **points of discontinuity**:

$$
\overline{f}(x_0) = \frac{1}{2}\left[ \lim_{x\to x^{-}_{0} } f(x)+\lim_{x\to x^+_0} f(x) \right], \quad -\frac{T}{2} < x_0 < \frac{T}{2} \tag{47}
$$

Note also that the *ringing* is commonly referred to as the [Gibbs phenomenon](https://en.wikipedia.org/wiki/Gibbs_phenomenon), which is an overshoot of the Fourier series (and other eigenfunction series) occurring at simple discontinuities.

Figure 4.13 contrasts the two kinds of discontinuity; at a jump, the series converges to the star.

In [ ]:
C = DSP_COLORS
xl, xr = np.linspace(-0.2, 1, 200), np.linspace(1, 2.2, 200)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
# (a) removable: both one-sided limits equal 1, but f(1) = 0
axa.plot(xl, xl**2, color=C['time_domain']); axa.plot(xr, 2 - xr, color=C['time_domain'])
axa.plot(1, 1, 'o', mfc='white', mec=C['time_domain'], ms=9, mew=2, zorder=5)
axa.plot(1, 0, 'o', color=C['time_domain'], ms=9, zorder=5)
axa.text(1.08, 1.06, r'$\lim_{x\to 1^\pm} f(x)=1$', fontsize=11)
axa.text(1.08, 0.05, '$f(1)=0$', fontsize=11)
axa.set_title('(a) Removable discontinuity')
# (b) jump: left limit 1, right limit 2; the Fourier series converges to their mean (star)
axb.plot(xl, xl**2, color=C['time_domain']); axb.plot(xr, 2 - (xr - 1)**2, color=C['time_domain'])
axb.plot(1, 1, 'o', mfc='white', mec=C['time_domain'], ms=9, mew=2, zorder=5)
axb.plot(1, 2, 'o', mfc='white', mec=C['time_domain'], ms=9, mew=2, zorder=5)
axb.plot(1, 0, 'o', color=C['time_domain'], ms=9, zorder=5)
fbar = 0.5*(1 + 2)
axb.plot(1, fbar, '*', color=C['freq_domain'], ms=16, zorder=6)
axb.text(0.15, 1.95, '$f(1^+)=2$', fontsize=11); axb.text(0.15, 1.0, '$f(1^-)=1$', fontsize=11)
axb.text(1.08, fbar - 0.05, rf'$\bar f(1)=\frac{{1}}{{2}}[f(1^-)+f(1^+)]$ = {fbar:.1f}', color=C['freq_domain'], fontsize=11)
axb.text(1.08, 0.05, '$f(1)=0$', fontsize=11)
axb.set_title('(b) Jump discontinuity')
for ax in (axa, axb):
    ax.set_xlim(-0.2, 2.2); ax.set_ylim(-0.2, 2.35); ax.set_xlabel('$x$')
axa.set_ylabel('$f(x)$')
plt.tight_layout(); plt.show()

<b>Figure 4.13.</b> (a) A removable discontinuity: both one-sided limits equal 1, but $f(1)=0$. (b) A jump discontinuity: the limits from the left (1) and right (2) differ. Open circles are the limits and filled circles the function values; the red star is the average of the two limits, equation (47), the value a Fourier series converges to at the jump. (Redrawn after Wikimedia Commons.) <i>(Python-generated.)</i>

Figure 4.14 adds the odd harmonics of a square wave of amplitude 1 one at a time, from $N=1$ to $N=59$. Watch the
overshoot next to each jump: as $N$ grows it moves toward the jump and gets narrower, but its height does not shrink; it settles near 9% of the
jump. Away from the jumps, the error does fall. This is the behavior QUESTIONS 1&ndash;4 ask you to find with the boxcar.

In [ ]:
C = DSP_COLORS
T = 2.0                                      # square wave of amplitude 1 and period 2 s (jump of 2 at t = +-0.5 s)
t = np.linspace(-1, 1, 4001)
tf = np.linspace(-1, 1, 400001)              # fine grid used only to measure the peak
square = np.sign(np.cos(np.pi*t))
Ns = list(range(1, 60, 2))                   # odd harmonics N = 1, 3, ..., 59 (30 frames)
kk = np.arange(1, 60, 2)
amp = 4/(np.pi*kk)                           # one-sided cosine amplitudes
sign = (-1)**((kk - 1)//2)
parts, peaks = [], []
for N in Ns:
    sel = kk <= N
    parts.append(np.sum((sign*amp)[sel][:, None]*np.cos(np.pi*kk[sel][:, None]*t), axis=0))
    pf = np.sum((sign*amp)[sel][:, None]*np.cos(np.pi*kk[sel][:, None]*tf[(tf > -0.5) & (tf < 0.5)]), axis=0)
    peaks.append(pf.max())
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw=dict(width_ratios=[2, 1]))
axa.plot(t, square, color=C['continuous'], lw=1.5)
line, = axa.plot([], [], color=C['time_domain'], lw=1.8)
axa.axhline(1, color=C['neutral'], ls=':', lw=1)
axa.set_xlim(-1, 1); axa.set_ylim(-1.45, 1.45); axa.set_xlabel('Time (s)'); axa.set_ylabel('Amplitude')
axa.set_title('(a) Square wave (gray) and partial sum (blue)', fontsize=11)
stems = axb.vlines([], [], [], color=C['freq_domain'], lw=2)
dots, = axb.plot([], [], 'o', color=C['freq_domain'])
axb.set_xlim(0, 60); axb.set_ylim(0, 1.4); axb.set_xlabel('Harmonic $k$'); axb.set_ylabel(r'$|A_k| = 4/(\pi k)$')
axb.set_title('(b) Harmonics included', fontsize=11)
title = fig.suptitle('', fontsize=13)

def update(i):
    N = Ns[i]; sel = kk <= N
    line.set_data(t, parts[i])
    stems.set_segments([[(k, 0), (k, a)] for k, a in zip(kk[sel], amp[sel])])
    dots.set_data(kk[sel], amp[sel])
    title.set_text(f'N = {N}: peak = {peaks[i]:.3f}, overshoot = {(peaks[i] - 1)/2*100:.1f}% of the jump')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=len(Ns), interval=300, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 4.14.</b> Partial Fourier sums of a square wave with amplitude 1 and period 2 s, using the odd harmonics up to $N=1,3,\dots,59$ (30 frames). (a) The square wave (gray) and the partial sum (blue). (b) The harmonic amplitudes $4/(\pi k)$ included so far. The peak falls from 1.273 ($N=1$) to 1.179 ($N=59$), an overshoot of 9.0% of the jump, which does not vanish as $N$ grows. <i>(Python-generated.)</i>

<div class="alert alert-success">
<h3><center>Distraction A: Michelson's machine and Gibbs's letter</center></h3>

In 1898 A. A. Michelson and S. W. Stratton built a mechanical harmonic
analyzer that could add 80 Fourier terms and draw the result. Its plots showed an overshoot next to every jump, which Michelson at first
suspected was a flaw in the machine. J. W. Gibbs explained the overshoot in letters to <i>Nature</i> in 1898&ndash;1899, and it later turned out that
H. Wilbraham had described it in 1848. The effect has carried Gibbs's name ever since.

<i>References:</i> A. A. Michelson and S. W. Stratton (1898), "A New Harmonic Analyzer," <i>American Journal of Science</i>, Series 4, 5(25), 1&ndash;13.
J. W. Gibbs (1898), "Fourier's Series," <i>Nature</i> 59, 200. J. W. Gibbs (1899), "Fourier's Series," <i>Nature</i> 59, 606.
H. Wilbraham (1848), "On a certain periodic function," <i>The Cambridge and Dublin Mathematical Journal</i> 3, 198&ndash;201.
</div>

**QUESTION:** 

Show that a periodic signal defined by

$$r(t) = \left\{
\begin{array}{l}
t+\frac{\pi}{2}, \quad \mathrm{if}\, -\pi < t<0 \\
-t+\frac{\pi}{2}, \quad \mathrm{if}\, 0 < t< \pi \\
\end{array}
\right. \tag{48}
$$

where $r(t+2\pi)=r(t)$ has the following Fourier series:

$$r(t) = \frac{4}{\pi} \sum_{n=1}^{\infty} \frac{\mathrm{cos} (2n-1)t}{(2n-1)^2}  \tag{49}$$ 

or, equivalently,

$$r(t) = \frac{4}{\pi} \sum_{n=1,odd}^{\infty}\frac{ \mathrm{cos} \, n t }{n^2}\quad n=1,3,5,...  \tag{50}$$ 

<a name="4.9"></a>
## 4.9 Fourier Series and ODEs/PDEs

Fourier Series have important applications in ODEs and PDEs, as we will demonstrate with an example.  It highlights how useful it is to split a periodic function into its simplest parts.

From previous (geo)physics classes we know that the forced oscillations of a body of mass $m$ on a spring of modulus $k$ are governed by the ODE:

$$ m\frac{d^2 y}{dt^2}+c\frac{dy}{dt}+ky = r(t) \tag{51}$$

where $y=y(t)$ is the displacement from rest, $c$ is the damping constant and $r(t)$ is the external force depending on time $t$.  If $r(t)$ is a sine or cosine function and if there is damping (i.e., $c>0$), then the steady-state solution is a harmonic oscillator with a frequency equal to that of $r(t)$.  However, if $r(t)$ is **not** a pure sine or cosine wave but is any other periodic function then the **steady state solution will be the superposition of harmonic oscillators with frequencies equal to that of $r(t)$ and its integer multiples**.

**Q:** Let's take the example in the QUESTION just above. What is the steady state solution $y(t)$ of forced oscillations under a driving force of  

$$r(t) = \left\{
\begin{array}{l}
t+\frac{\pi}{2}, \quad \mathrm{if}\, -\pi < t<0 \\
-t+\frac{\pi}{2}, \quad \mathrm{if}\, 0 < t< \pi \\
\end{array}
\right.\tag{52}
$$

**A:** From the previous question we know that its Fourier series is:

$$r(t) = \frac{4}{\pi} \sum_{n=1,odd}^{\infty}\frac{ \mathrm{cos} \, n t }{n^2}\quad n=1,3,5,...  \tag{53}$$ 

We know that the steady-state solution of equation (51) is of the form:

$$y_n = A_n \mathrm{cos}\,nt +  B_n \mathrm{sin}\,nt \tag{54} $$

Thus we can show that by substituting equation (54) into equation (51) we obtain coefficients:

$$A_n = \frac{4(k-mn^2)}{n^2\pi D_n}\quad \mathrm{and} \quad B_n=\frac{4c}{n\pi D_n}, \quad \mathrm{where} \quad D_n = (k-m n^2)^2+(cn)^2 \tag{55}$$

Since the ODE is linear, we expect the solution to be given by 

$$y = y_1+y_3+y_5+ ... \tag{56}$$

<div class="alert alert-info">
<b>Synthesis &mdash; Fourier series solve linear ODEs and PDEs.</b> In MATH 225 (and PHGN 100) you found the steady response of
$my''+cy'+ky=F_0\cos\omega t$ by undetermined coefficients. Because the equation is linear, a periodic forcing written as a Fourier series produces
a response that is the sum of those single-frequency answers, harmonic by harmonic: equation (55). The harmonic nearest $\sqrt{k/m}$
dominates, so the oscillator acts as a frequency-selective filter. The same orthogonality integrals, equations (13)&ndash;(22), are what
you used in POTE1 (GPGN 328) after separation of variables, where each separated mode is multiplied by one Fourier coefficient chosen to fit the
boundary condition.
</div>

Figure 4.15 plots an example where $m$=1, $c=$0.05 and $k$=25:

In [ ]:
C = DSP_COLORS
m, c, k = 1, 0.05, 25                        # mass, damping, spring constant (natural frequency sqrt(k/m) = 5 rad/s)
nc = 25
t = np.linspace(-5*np.pi, 5*np.pi, 6001)
n = np.arange(1, nc, 2)                      # odd harmonics of the triangle forcing, eq. (47)
Dn = (k - m*n**2)**2 + (c*n)**2
An = 4*(k - m*n**2)/(n**2*np.pi*Dn)
Bn = 4*c/(n*np.pi*Dn)
y = np.sum(An[:, None]*np.cos(n[:, None]*t) + Bn[:, None]*np.sin(n[:, None]*t), axis=0)
Rn = np.sqrt(An**2 + Bn**2)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw=dict(width_ratios=[1.6, 1]))
axa.plot(t, y, color=C['output'])
axa.set_xlim(t[0], t[-1]); axa.set_xlabel('Time (s)'); axa.set_ylabel('$y(t)$'); axa.set_title('(a) Steady-state response')
axb.vlines(n, 0, Rn, color=C['freq_domain'], lw=2); axb.plot(n, Rn, 'o', color=C['freq_domain'])
axb.set_xlabel('Harmonic $n$'); axb.set_ylabel(r'$R_n=\sqrt{A_n^2+B_n^2}$'); axb.set_title('(b) Response amplitude per harmonic')
axb.set_xlim(0, nc); axb.set_ylim(0, 1.15*Rn.max())
plt.tight_layout(); plt.show()

<b>Figure 4.15.</b> Forced, damped oscillation of a mass $m=1$ on a spring $k=25$ with damping $c=0.05$, driven by the triangle wave of equation (52). (a) The steady-state response. (b) The response amplitude $R_n=\sqrt{A_n^2+B_n^2}$ of each odd harmonic: $R_5=0.2037$ dwarfs $R_1=0.0531$ because $n=5$ sits at the natural frequency $\sqrt{k/m}=5$ rad/s, so the response is nearly a pure $n=5$ sinusoid. <i>(Python-generated.)</i>

<a name="4.10"></a>
## 4.10 The Heartbeat Revisited

At the very start of this section we looked at the heartbeat signal.  Let's now look at what type of frequencies are in this using the Scipy toolbox's **periodogram** function.  Essentially, this is a tool that allows us to look at the frequency structure of discrete (and ideally periodic!) functions.

In [ ]:
C = DSP_COLORS
tt = np.arange(len(xx))/sr
f, Pxx = signal.periodogram(xx, sr)
f0 = 1.25                                    # measured beat rate (Hz): one beat every 0.8 s
fig, axs = plt.subplots(2, 2, figsize=(13, 8))
axs[0, 0].plot(tt, xx, color=C['time_domain'], lw=0.5)
axs[0, 0].set_xlim(0, tt[-1]); axs[0, 0].set_xlabel('Time (s)'); axs[0, 0].set_ylabel('Amplitude')
axs[0, 0].set_title('(a) Heartbeat record (75 bpm, 1.25 beats per second)', fontsize=12)
n1 = int(0.73*sr)
axs[0, 1].plot(tt[:n1], xx[:n1], color=C['time_domain'], lw=0.7)
axs[0, 1].set_xlim(0, tt[n1]); axs[0, 1].set_xlabel('Time (s)'); axs[0, 1].set_ylabel('Amplitude')
axs[0, 1].set_title('(b) One heartbeat (lub–dub)', fontsize=12)
for ax, fmax, ttl in ((axs[1, 0], 10, '(c) Periodogram, 0–10 Hz'), (axs[1, 1], 300, '(d) Periodogram, 0–300 Hz')):
    ax.semilogy(f, Pxx, color=C['freq_domain'], lw=0.8)
    ax.set_xlim(0, fmax); ax.set_ylim(1e-13, 1e2)
    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (1/Hz)'); ax.set_title(ttl, fontsize=12)
for kf in np.arange(1, 9)*f0:
    axs[1, 0].axvline(kf, color=C['neutral'], ls=':', lw=1)
plt.tight_layout(); plt.show()

<b>Figure 4.16.</b> The heartbeat of Figure 4.1. (a) The full record; counting beats gives about 75 beats per minute, i.e., 1.25 beats per second or one beat every 0.8 s. (b) One heartbeat, showing the "lub-DUB" pair: the "lub" is the sound of the tricuspid and mitral valves closing, and the "DUB" that of the aortic and pulmonary valves closing; note the many high-frequency components. (c) The <i>periodogram</i> estimate of the power spectral density, zoomed to 0&ndash;10 Hz: the lowest contributing frequency is 1.25 Hz, the heart rate, and the dotted lines mark its first eight multiples, the first eight Fourier coefficients of one beat. (d) The periodogram out to 300 Hz. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; geophysical data as the running example.</b> Real data are only nearly periodic. The heartbeat's lines at multiples of
1.25 Hz are broadened because the beat interval wanders slightly, so a periodogram of a long record smears each $C_k$ into a narrow peak. The same
holds for geophysical "periodic" signals: tidal gravity, powerline noise at 60 Hz and its harmonics, and the base frequency of an EM transmitter
(Microquestion 4.1).
</div>

<a name="4.11"></a>
## 4.11 Parting Thoughts

Examining Fourier Series is a good introduction to thinking about the differences in the time and frequency domain (well ... conjugate Fourier domains in general).  It is important to emphasize that the harmonic analysis you saw here is for examining **periodic** and **continuous-time** functions and looking at their **discrete** Fourier components, which represent a sequence.  

Because many of the signals we examine are both **discrete** and of **finite** duration, we need additional tools that account for these two key differences: Module 5 lets $T\to\infty$ to obtain the Fourier transform, and Module 11 returns to finite, sampled records with the Discrete Fourier Transform (DFT).

In addition to their usefulness in solving ODEs and PDEs, you will likely encounter Fourier Series when using symbolic analysis using packages like Mathematica, MATLAB (Symbolic Math Toolbox) and indeed Python (SymPy).  You will also use them when designing specific output waveforms that are used in exploration geophysics instrumentation.

### References

1. Arfken, G. "Fourier Series." Ch. 14 in Mathematical Methods for Physicists, 3rd ed. Orlando, FL: Academic Press, pp. 760-793, 1985.

2. Boas, M.L.,  Mathematical Methods in the Physical Science, John Wiley, 2006.

3. Byerly, W. E. An Elementary Treatise on Fourier's Series, and Spherical, Cylindrical, and Ellipsoidal Harmonics, with Applications to Problems in Mathematical Physics. New York: Dover, 1959.

4. Davis, H. F. Fourier Series and Orthogonal Functions. New York: Dover, 1963.

5. James, J.F. 2011, A Student's Guide to Fourier Transforms, 3rd ed, Cambridge University Press.

6. Whittaker, E. T. and Robinson, G. "Practical Fourier Analysis." Ch. 10 in The Calculus of Observations: A Treatise on Numerical Mathematics, 4th ed. New York: Dover, pp. 260-284, 1967.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 4.1 &mdash; The spectrum of a TEM transmitter</b>

A time-domain EM transmitter drives the current waveform used in many TEM and IP surveys: $+I_0$ for a quarter period, off for a quarter period, $-I_0$ for a quarter period, and off for a quarter period, with $I_0 = 10$ A and base frequency $f_0 = 25$ Hz ($T = 40$ ms). Take $t=0$ at the start of the positive pulse, and write the current as $I_0\,x(t)$ with $x(t) \in \{1, 0, -1\}$.

<b>(a)</b> Use equation (4) to find $C_0$, $C_1$, $C_2$ and $C_3$ of $x(t)$. Give the one-sided cosine amplitude $2|C_k| I_0$ (in amperes) and the frequency of the first three nonzero harmonics.

<b>(b)</b> Using the orthogonality relations (equations 13&ndash;22), show that the mean power $\frac{1}{T}\int_0^T x^2\,dt$ equals $\sum_k |C_k|^2$. What fraction of the power is in the 25 Hz fundamental? How many odd harmonics are needed to carry 95%? (You may use $\sum_{k\ \mathrm{odd}} 1/k^2 = \pi^2/8$, MATH 213.)

<b>(c)</b> A student argues: "the waveform is off half the time, so it must contain a DC term, and the even harmonics (50 and 100 Hz) must be present." Find the flaw, and say why a 25 Hz base frequency is a sensible choice for a survey near 50 Hz powerlines.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 4.1</b></summary>

<b>(a)</b> $C_k = \frac{1}{T}\left[\int_0^{T/4} e^{-ik\omega_0 t}dt - \int_{T/2}^{3T/4} e^{-ik\omega_0 t}dt\right] = \frac{(1-e^{-ik\pi})(1-e^{-ik\pi/2})}{2\pi i k}$. $C_0 = 0$ (equal positive and negative on-times) and $C_2 = 0$ (the factor $1 - e^{-i2\pi} = 0$). $C_1 = (1-i)/\pi = 0.3183 - 0.3183i$, so $|C_1| = \sqrt{2}/\pi = 0.4502$ with phase $-45^\circ$; $C_3 = -(1+i)/(3\pi)$, $|C_3| = 0.1500$, phase $-135^\circ$. In general $|C_k| = \sqrt{2}/(\pi k)$ for odd $k$. Amplitudes $2|C_k| I_0$: 9.00 A at 25 Hz, 3.00 A at 75 Hz, 1.80 A at 125 Hz.

<b>(b)</b> Substitute equation (3) into $\frac{1}{T}\int x^2\,dt$ (with $x$ real, $x^2 = x\bar{x}$); orthogonality removes every cross term with $k \neq l$, leaving $\sum |C_k|^2$. The mean power is $\frac{1}{2}$ (on half the time, with $x^2 = 1$). Fundamental: $2|C_1|^2/\frac{1}{2} = 8/\pi^2 = 81.1\%$. Cumulative: 90.1% ($k \le 3$), 93.3% ($k \le 5$), 94.96% ($k \le 7$), and 96.0% with five odd harmonics ($k \le 9$, up to 225 Hz). Check: $\sum_{\mathrm{odd}} 4/(\pi^2 k^2) = \frac{4}{\pi^2}\cdot\frac{\pi^2}{8} = \frac{1}{2}$.

<b>(c)</b> $C_0$ is the average, which is zero because the positive and negative pulses cancel. The waveform has half-wave symmetry, $x(t+T/2) = -x(t)$, which gives the factor $1 - e^{-ik\pi}$ in (a); it is zero for every even $k$. So the transmitter puts no energy at 50 Hz or 100 Hz. With $f_0 = 25$ Hz, powerline noise at 50 Hz and its harmonics falls on even harmonics, where the transmitter signal is zero, so it can be separated from the signal (compare Figure 4.10, where only odd and zero harmonics appear).
</details>

<div class="alert alert-warning">
<b>Microquestion 4.2 &mdash; When one harmonic takes over</b>

Use the driven oscillator of equation (51) with the triangle-wave forcing $r(t)$ of equation (52) (period $2\pi$ s), but with $m = 1$ kg, $k = 9$ N/m and $c = 0.2$ kg/s.

<b>(a)</b> Which harmonic $n$ of $r(t)$ lies at the undamped natural frequency $\sqrt{k/m}$? Using equation (55), compute the response amplitude $R_n = \sqrt{A_n^2 + B_n^2}$ for $n = 1$ and $n = 3$, and compare each with the forcing amplitude $4/(\pi n^2)$.

<b>(b)</b> The forcing at $n = 3$ is 9 times weaker than at $n = 1$. Show that the response at $n = 3$ is still about 1.5 times larger, and find the phase of $y_3$ relative to the $\cos 3t$ forcing (MATH 225).

<b>(c)</b> A geophone is a mass on a spring (PHGN 100). A student records a periodic ground motion with a lightly damped sensor whose natural frequency lies inside the signal band, sees an almost pure sinusoid, and concludes the ground motion was sinusoidal. What is wrong, and what design choice avoids the problem?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 4.2</b></summary>

<b>(a)</b> $\sqrt{k/m} = 3$ rad/s, so $n = 3$. $D_n = (k - mn^2)^2 + (cn)^2$: $D_1 = 64.04$, $D_3 = 0.36$. $R_n = 4/(\pi n^2 \sqrt{D_n})$: $R_1 = 0.1591$ (forcing 1.2732, gain 0.125) and $R_3 = 0.2358$ (forcing 0.1415, gain 1.667).

<b>(b)</b> $R_3/R_1 = 1.48$. At $n = 3$, $k - mn^2 = 0$, so $A_3 = 0$ and $B_3 = 4c/(3\pi D_3) = 0.2358$: the response is $0.236 \sin 3t$, a $90^\circ$ lag behind the $\cos 3t$ forcing, the standard resonance phase from MATH 225 and PHGN 100.

<b>(c)</b> The oscillator amplifies the harmonic near resonance and suppresses the rest ($R_5 = 0.0032$, $R_7 = 0.0006$), so the output is nearly a single sinusoid even though the input was a triangle wave (compare Figure 4.15). The output shape tells you about the sensor as much as the ground. Geophones are built with their natural frequency below the band of interest and with damping near 0.7 of critical, so the response is flat across the band and no single harmonic dominates.
</details>

<div class="alert alert-warning">
<b>Microquestion 4.3 &mdash; Moving the start of the record</b>

Treat the heartbeat of Figure 4.1 as exactly periodic with $T = 0.80$ s (75 beats per minute), with coefficients $C_k$ computed from a record that starts at $t = 0$.

<b>(a)</b> What is the line spacing $f_0$ in hertz, and how many nonzero-frequency lines lie between 0 and 10 Hz?

<b>(b)</b> The same heartbeat is recorded, but the recording starts $\tau = 0.2$ s later, so $x_\mathrm{new}(t) = x(t+\tau)$. Using equation (4) and a change of variable (MATH 213), show that $C_k^\mathrm{new} = C_k e^{ik\omega_0\tau}$. What happens to $|C_k|$, and by how many degrees does the phase of $C_1$, $C_2$ and $C_3$ change? (Module 3: $|e^{i\theta}| = 1$.)

<b>(c)</b> "If the heart rate rises to 90 bpm, every spectral line moves up by 0.25 Hz." Correct the statement using the squeeze/stretch discussion in Section 4.3.1.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 4.3</b></summary>

<b>(a)</b> $f_0 = 1/T = 1.25$ Hz; lines at 1.25, 2.50, …, 10.0 Hz: 8 lines, consistent with Figure 4.16(c).

<b>(b)</b> $C_k^\mathrm{new} = \frac{1}{T}\int_{-T/2}^{T/2} x(t+\tau) e^{-ik\omega_0 t}\,dt$. With $u = t + \tau$, $e^{-ik\omega_0 t} = e^{-ik\omega_0 u} e^{ik\omega_0\tau}$, and the shifted limits still span one full period of a periodic integrand, so $C_k^\mathrm{new} = e^{ik\omega_0\tau} C_k$. The magnitude is unchanged. The phase changes by $+k\omega_0\tau = 2\pi k\tau/T$: $+90^\circ$, $+180^\circ$ and $+270^\circ$ ($\equiv -90^\circ$) for $k = 1, 2, 3$. The phase change grows linearly with $k$.

<b>(c)</b> At 90 bpm, $T = 0.667$ s and $f_0 = 1.5$ Hz. Line $k$ sits at $k f_0$, so it moves by $k \times 0.25$ Hz: 0.25, 0.50, 0.75 and 1.00 Hz for $k = 1$ to 4. A shorter period stretches the whole line pattern; it does not slide it.
</details>